# Agent Framework Lab: Models, Tools, and a SQLite Worker

**Framework clarification:** although this file is named `agno_lab.ipynb`, its code imports `agent_framework` and uses Microsoft Agent Framework. It does not demonstrate Agno's `Agent` or model APIs. The existing filename and code are unchanged.

This reference progresses from a single model response to a worker that plans a translation task, reads and writes files through MCP, and records completion in SQLite.

| Stage | Purpose | Main APIs |
|---|---|---|
| Setup | Configure warnings and load environment settings | `warnings.filterwarnings`, `load_dotenv` |
| Steps 1-2 | Configure an OpenAI client and run an agent | `OpenAIChatClient`, `Agent`, `agent.run`, `result.text` |
| Todo board | Seed and display persistent goals and steps | Local `board.py` helpers |
| Step 3 | Expose Python operations to the model | Functions in `tools=[...]` |
| Step 4 | Connect a filesystem MCP server | `MCPStdioTool`, `stdio_client`, `async with filesystem` |
| Step 5 | Combine planning, file tools, and progress tracking | Worker agent with board and filesystem tools |

**Prerequisites:** the project's Python environment with Microsoft Agent Framework, its OpenAI integration, MCP, and python-dotenv; `OPENAI_API_KEY`; the local `board.py`; Node.js/`npx`; and an existing `workspace/notes.txt`. Run with this project directory as the kernel's working directory for the intended relative paths.

**Side effects:** `.env` values can override existing environment settings. Both board resets delete existing todos in the configured database. Model calls can incur charges, `npx -y` can download a server package, and the final worker can overwrite `tagalog.txt` through its filesystem tools.

**Review scope:** explanations and comments only. No model calls, MCP server launches, or database/file-mutating notebook cells were rerun. Saved outputs are historical evidence of the happy path, not an independent verification of translation quality.

In [1]:
# The framework prints a couple of "experimental" notices on import; quiet them.
import warnings
# Warning message filters match from the start, not anywhere in the text.
warnings.filterwarnings("ignore", message=r"experimental.*")

In [3]:
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from agent_framework import Agent, MCPStdioTool
from agent_framework.openai import OpenAIChatClient 

# Load .env values, replacing matching environment variables when present.
load_dotenv(override=True)

True

## Step 1: Create The Agent

`OpenAIChatClient(model=MODEL)` configures the model client; `Agent(client=client, instructions=...)` gives the agent its behavior. The same client is reused by later agents, but each agent has its own instructions and tools. These are Microsoft Agent Framework APIs, not Agno APIs.

In [4]:
MODEL = "gpt-5.4-mini" 

client = OpenAIChatClient(model=MODEL)

agent = Agent(
    client=client,
    instructions="You're an concise. friendly assistant. Reply in a single short sentence.", 
)

## Step 2: Run It

`await agent.run(...)` waits asynchronously for a run. `result.text` is its text response, printed explicitly here. Jupyter supports top-level `await`; a plain script needs an asynchronous entry point. The saved response demonstrates the simple, tool-free greeting.

In [5]:
result = await agent.run("Say hello in Tagalog.")
print(result.text)

Kumusta!


# Project: SQLite Todo Board

`board.py` stores todos in SQLite. Goals have `parent_id=None`; child steps reference a goal ID. New rows are `pending`, claiming sets `in_progress`, and completing sets `done` with a result summary. The default database is beside `board.py`, unless `BOARD_PATH` was set before importing that module. This task store is not a conversation-memory implementation.

In [6]:
import board

# CAUTION: drop the existing todos table and create an empty board.
board.reset_board()
board.add_goal("Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.',
  'status': 'pending',
  'result': ''}]

In [7]:
# Rich-formatted display for humans; the database rows are not changed.
board.show_board()

Goal #1: Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.

## Step 3: Add Tools

The framework accepts Python callables in `tools`. Type annotations and docstrings describe the callable inputs and purpose. `show_todos` reads rows, `plan_steps` adds child rows, and `complete_task` updates status and result text.

`plan_steps` uses a list comprehension, so its returned IDs preserve the input step order and are JSON-compatible. The introductory `board_agent` registers listing and completion but not planning; planning is added in Step 5. Its status request is read-oriented, but the agent still has a mutating completion tool.

In [10]:
# Structured rows are useful to the model; show_board() is for human display.
def show_todos()-> list[dict]:
    """ List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id. """
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """ Break a goal into an ordered checklist of steps on the board. Pass the goals id and a short list of step descriptions."""
    return {
        'goal_id': goal_id,
        # Each insert returns an ID; the list preserves the supplied step sequence.
        'step_ids': [board.add_step(goal_id, step) for step in steps]
    }

def complete_task(task_id: int, result: str) -> dict:
    """ Mark a todo (a step or the goal) with this id as done and record a short result summary. """
    # The helper does not check whether this ID exists or child tasks are finished.
    board.complete_todo(task_id, result)
    return {
        'task_id': task_id,
        'status': 'done' 
    }

In [11]:
board_agent = Agent(
    client=client,
    instructions="You help manage a shared todo board.",
    tools=[show_todos,
           complete_task]
)

In [12]:
result = await board_agent.run("What is on the board right now, and what is its status?")
print(result.text)

There is 1 item on the board:

- **Task 1** — *Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.*  
  **Status:** pending




## Step 4: Add MCP

`FilesystemMCP` customizes the SDK's stdio-client factory. It passes a workspace working directory to the subprocess and requests suppression of server stderr. The factory returns a context manager; constructing the object alone is not a filesystem operation.

The server receives the resolved workspace as its allowed file root. This governs the filesystem tools, not the Python kernel or the separate SQLite database. `Path("workspace").resolve()` depends on the kernel's working directory and does not create the folder.

`async with filesystem` opens the MCP connection for the agent run and closes it on exit, including exceptional exits. The read-only example prints its response after leaving that context. The same tool object is used in a later, separate context for the worker.

In [13]:
# Resolve relative to the kernel's current working directory; the folder must exist.
workspace = Path("workspace").resolve()

class FilesystemMCP(MCPStdioTool):
    """ The filesystem server with its stderr sent to DEVNULL and its working
    directory set to the workspace, so file names resolve there and it runs
    cleanly from a Jupyter kernel on Windows."""

    def get_mcp_client(self):
        from mcp.client.stdio import StdioServerParameters, stdio_client

        # This override reads the global workspace when the connection is opened.
        params = StdioServerParameters(command=self.command, args=self.args, env=self.env, cwd=str(workspace))
        return stdio_client(server=params, errlog=subprocess.DEVNULL)

filesystem = FilesystemMCP(
    name="filesystem",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],   
)

In [14]:
file_agent = Agent(
    client=client,
    instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem]
)

In [15]:
# Keep the MCP connection open for the full agent run.
async with filesystem:
    result = await file_agent.run("Read notes.txt and summarize it in one short sentence.")
print(result.text)

The notes say the team is building a small language tutor one task at a time, with each helper carefully completing and marking off their assigned work.


## Step 5: Put It In A Loop With A Goal

The worker combines listing, planning, completion, and file tools. The framework manages the model/tool iteration inside `worker.run`; there is no explicit Python loop in this cell.

The cell resets the board again, seeds and claims the goal, then runs the worker inside the filesystem connection context. The saved board display records one completed goal and three completed steps. Those statuses are tool updates, not independent checks of the translation file. The final run result is discarded rather than printed or assigned.

In [16]:
INSTRUCTIONS = """ 
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan:
the handful of concrete steps the work itself breaks down into, added to the
board under the goal. Then carry them out with your file tools, marking each 
step done as you finish it. Once the steps are all done, close the goal. Your
files live in the single folder your tools are allow to  use. 
"""

worker = Agent(
    client=client,
    instructions=INSTRUCTIONS,
    tools=[
        show_todos,
        plan_steps,
        complete_task,
        filesystem
    ]
)

# Destructive reseeding: this replaces the earlier demonstration's todos.
board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.")
# Claiming changes the status to in_progress, despite the pending-goal prompt.
board.claim_todo(goal_id)

async with filesystem:
    # The returned run result is not assigned; result still refers to the summary run.
    await worker.run("Please work the pending goal on the board.")
board.show_board()

Goal #1: Read notes.txt, translate its contents into Tagalog, and write the Tagalog to tagalog.txt.  Completed translation of notes.txt into tagalog.txt.
  Step #2: Read notes.txt and capture its contents  Read notes.txt and captured its contents for translation.
  Step #3: Translate the notes into Tagalog  Translated the notes into Tagalog.
  Step #4: Write the Tagalog translation to tagalog.txt  Wrote the Tagalog translation to tagalog.txt.

## Conclusion and Reference Summary

This notebook demonstrates Microsoft Agent Framework's progression from a model client and instructions to Python tools, an MCP filesystem connection, and a goal-driven worker. Its basic orchestration is coherent, and saved outputs show the examples reached a completed board state.

- **Configuration:** the client selects the model; each agent selects its own instructions and tools.
- **Execution:** `await agent.run(...)` returns a response; `print(result.text)` displays it.
- **Tasks:** SQLite persists goals, steps, statuses, and summaries. A claimed goal is `in_progress`.
- **External tools:** `async with filesystem` manages the MCP connection while the agent works.
- **Planning:** the current `plan_steps` correctly returns an ordered list of IDs.

**Evidence and limits:** an isolated test using a fake board confirmed insertion order and JSON-compatible planning output. Inspection of the installed MCP/Agent Framework APIs confirmed the stdio-client hook and exposed the override's dependency on SDK details. The installed framework emitted `[SKILLS]` and `[HARNESS]` experimental warnings, whose prefixes do not match `experimental.*`. No fresh end-to-end run or translation-quality validation was performed; suggestions below identify remaining reliability concerns.

## Tips to Improve Code Logic - Not Applied

These are proposals only. Executable statements, prompts, docstrings, and existing outputs have not been changed.

| Tip | Current issue and implication of a future change |
|---|---|
| Clarify the framework identity | The filename says Agno, but the imports use Microsoft Agent Framework. Renaming or relabeling the notebook would improve reference accuracy; migrating to actual Agno would require substantive API changes, not a cosmetic import swap. |
| Pass the claimed goal ID explicitly | The code marks the goal `in_progress`, while the worker instructions and request ask for a pending goal. An ID-specific request describing the claimed task would remove conflicting status wording and prevent ambiguous goal selection. |
| Validate task existence and prerequisites | `add_step` does not check that its parent is a real goal; `complete_todo` can update zero rows, while the tool still returns `done`. Add clear errors for invalid IDs and prevent closing a goal with unfinished children. This would deliberately reject calls currently accepted. |
| Verify the produced artifact before completion | Read back the expected file and check it exists and contains content before closing the goal. This catches missing/empty output, but translation accuracy still requires a suitable review. |
| Protect board data during reruns | Both resets drop all existing todos. Use an isolated demo database or require an intentional reset step to avoid deleting unrelated work. This changes the storage location or rerun procedure. |
| Check prerequisites before paid calls and resets | Verify the API key, `npx`, the intended workspace, and `notes.txt` first. Use an explicit project-root strategy if running from different working directories; fail clearly rather than selecting an unintended folder. |
| Preserve diagnostics instead of broadly quieting them | Warning filters match from the start, so `experimental.*` does not match the observed `[SKILLS]` and `[HARNESS]` messages. Prefer leaving warnings visible, or use a narrowly scoped, verified category/message filter if intentional. Suppressing server stderr also removes useful startup diagnostics; retain logs when troubleshooting. |
| Reduce reliance on a custom SDK hook | The override bypasses the base factory's encoding and client-keyword forwarding, and it reads a global workspace at connection time. Prefer supported client options where available; otherwise preserve those options and store the workspace per instance. This changes connection setup, so test against the installed SDK. The installed `stdio_client` annotates `errlog` as `TextIO`; the current code passes the integer `subprocess.DEVNULL`, so portability should be verified rather than inferred from one saved run. |
| Narrow the status agent's permissions | Remove `complete_task` from `board_agent` if it should only report status. This would intentionally remove its ability to update the board; adding `plan_steps` instead is only needed if it should plan tasks. |
| Capture the final worker response | Assign and inspect the final run result. The current `result` remains the earlier file-summary result. Capturing it improves debugging without changing the requested translation. |
| Correct model-facing wording | The introductory prompt says `You're an concise. friendly assistant`, and the worker says `are allow to  use`. Correcting these improves clarity but changes text sent to the model, so it remains a separate code change. |
| Make repeated runs reproducible | Pin a tested filesystem-server version instead of unversioned `npx -y`, and define whether rerunning a goal should replace output or resume previous steps. Consider a transaction for multi-step plan insertion, since the current helper commits each step separately and can leave a partial plan after failure. These change update and recovery behavior. |

**Unchanged:** model configuration, tool implementations, SDK subclass, warning filter, environment loading, database operations, task prompts, and file operations.